# Video Editing with Layout-Guided Attention

SDEdit + layout-guided optimization + TTM-style source blending.
The model is loaded once; re-run config and run cells to iterate.

In [ ]:
import logging
import os
import sys
from pathlib import Path

import torch
from diffusers.callbacks import MultiPipelineCallbacks, PipelineCallback
from diffusers.utils import export_to_video, load_video
from diffusers.utils.torch_utils import randn_tensor
from loguru import logger

PROJECT_ROOT = Path(os.getcwd()).resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.optimization import (
    AttentionStore,
    GuidingPhrase,
    LatentRefiner,
    LayoutKLLoss,
    get_token_indices_for_phrase_indices,
)
from src.video_utils import (
    SaveAttnMapsCallbackWAN,
    downsample_mask_to_latent_resolution,
    load_video_as_tensor,
    video_to_tensor,
)
from src.wan.attn_processor import (
    register_attention_hook,
    unregister_attention_hook,
)
from src.wan.pipeline import LayoutGuidedWanPipeline, encode_video_to_latents, load_pipeline

logging.getLogger("imageio_ffmpeg").setLevel(logging.ERROR)
logging.getLogger("imageio").setLevel(logging.ERROR)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

DATA_PATH = os.path.join(PROJECT_ROOT, "data")
OUTPUTS_PATH = os.path.join(PROJECT_ROOT, "outputs")

pipe = load_pipeline()

# Source Blending & Editing Functions

In [ ]:
class SourceBlendingCallback(PipelineCallback):
    """Replaces preserved regions with re-noised source latents at each step.

    At each blending step, source latents are re-noised to the next timestep's
    level using fixed noise, then blended into preserved regions (mask=0).

    Args:
        source_latents: Clean encoded source video latents.
        mask: Binary mask. 1 = edit region, 0 = preserved region.
        denoise_from: Step index where denoising starts in the full schedule.
        blend_start: First step_index to apply blending (inclusive).
        blend_stop: Last step_index to apply blending (exclusive).
        generator: RNG for the fixed noise tensor.
    """

    def __init__(self, source_latents, mask, denoise_from, blend_start, blend_stop, generator=None):
        super().__init__()
        self.source_latents = source_latents
        self.mask = mask
        self.denoise_from = denoise_from
        self.blend_start = blend_start
        self.blend_stop = blend_stop
        self.fixed_noise = randn_tensor(
            source_latents.shape,
            generator=generator,
            device=source_latents.device,
            dtype=source_latents.dtype,
        )

    @property
    def tensor_inputs(self):
        return ["latents"]

    def callback_fn(self, pipeline, step_index, timestep, callback_kwargs):
        if step_index < self.blend_start or step_index >= self.blend_stop:
            return callback_kwargs

        latents = callback_kwargs["latents"]
        timesteps = pipeline.scheduler.timesteps
        next_idx = self.denoise_from + step_index + 1

        if next_idx < len(timesteps):
            t = torch.as_tensor(timesteps[next_idx], device=self.source_latents.device, dtype=torch.long)
            noisy_source = pipeline.scheduler.add_noise(self.source_latents, self.fixed_noise, t.view(1)).to(
                dtype=latents.dtype, device=latents.device
            )
        else:
            noisy_source = self.source_latents.to(dtype=latents.dtype, device=latents.device)

        callback_kwargs["latents"] = latents * self.mask + noisy_source * (1 - self.mask)
        return callback_kwargs


def setup_layout_guidance(
    bbox_paths,
    phrase_indices,
    prompt,
    tokenizer,
    height,
    width,
    num_frames,
    num_refinements,
    lr_start,
    lr_end,
    vae_scale_factor_temporal=4,
    vae_scale_factor_spatial=8,
):
    """Build layout-guidance components: guiding phrases, bounding boxes, loss, and refiner."""
    indices = get_token_indices_for_phrase_indices(phrase_indices, prompt, tokenizer)
    words = prompt.split()
    guiding_phrases = [GuidingPhrase(phrase=" ".join(words[i] for i in group), indices=idx) for group, idx in zip(phrase_indices, indices)]
    logger.info(f"Guiding phrases: {guiding_phrases}")

    bboxes = torch.cat([load_video_as_tensor(p, height=height, width=width, mode="L")[:, :, :num_frames] for p in bbox_paths])
    bboxes = downsample_mask_to_latent_resolution(bboxes, vae_scale_factor_temporal, vae_scale_factor_spatial * 2)
    bboxes = (bboxes > 0.5).float()

    refiner = LatentRefiner(
        loss_fn=LayoutKLLoss(bboxes=bboxes, indices=indices, eps=1e-8),
        num_refinements=num_refinements,
        lr_start=lr_start,
        lr_end=lr_end,
        betas=(0.9, 0.999),
    )
    return guiding_phrases, refiner


def edit_video(
    pipe,
    prompt,
    negative_prompt,
    source_video_path,
    mask_video_path,
    height,
    width,
    num_frames,
    num_inference_steps,
    guidance_scale,
    guidance_scale_2,
    seed,
    noise_step,
    guiding_phrases,
    refiner,
    output_path,
    blend_start=0,
    blend_stop=None,
    layers_to_hook=None,
    refine_steps=None,
):
    """Edit a source video using SDEdit + layout guidance + source blending."""
    layers_to_hook = layers_to_hook or [10, 11, 12]
    refine_steps = refine_steps or []

    source_frames = [f.resize((width, height)) for f in load_video(source_video_path)[:num_frames]]
    source_latents = encode_video_to_latents(pipe, video_to_tensor(source_frames))

    edit_mask = load_video_as_tensor(mask_video_path, height=height, width=width, mode="L")[:, :, :num_frames]
    edit_mask = downsample_mask_to_latent_resolution(
        edit_mask,
        pipe.vae_scale_factor_temporal,
        pipe.vae_scale_factor_spatial,
    )
    edit_mask = (edit_mask > 0.5).float().to(device=source_latents.device, dtype=source_latents.dtype)

    generator = torch.Generator(device=pipe.device).manual_seed(seed)
    noise = randn_tensor(source_latents.shape, generator, device=pipe.device, dtype=source_latents.dtype)
    pipe.scheduler.set_timesteps(num_inference_steps)
    t = torch.as_tensor(pipe.scheduler.timesteps[noise_step], device=pipe.device, dtype=torch.long).view(1)
    noised_latents = pipe.scheduler.add_noise(source_latents, noise, t.long())

    num_denoise_steps = num_inference_steps - noise_step
    if blend_stop is None:
        blend_stop = num_denoise_steps
    logger.info(f"SDEdit from step {noise_step} (t={t.item()}), {num_denoise_steps} steps, blend=[{blend_start}, {blend_stop})")

    store = AttentionStore()
    attn_processors = register_attention_hook(pipe, store, layers_to_hook=layers_to_hook)

    lat_shape = (num_frames, height // (pipe.vae_scale_factor_spatial * 2), width // (pipe.vae_scale_factor_spatial * 2))
    attn_callback = SaveAttnMapsCallbackWAN(
        store=store,
        guiding_phrases=guiding_phrases,
        latent_shape=lat_shape,
        patch_size=pipe.transformer.config["patch_size"],
        output_path=str(Path(output_path) / "attention_maps"),
        fps=3.0,
    )

    blend_callback = SourceBlendingCallback(
        source_latents=source_latents,
        mask=edit_mask,
        denoise_from=noise_step,
        blend_start=blend_start,
        blend_stop=blend_stop,
        generator=torch.Generator(device=pipe.device).manual_seed(seed + 1),
    )

    generator = torch.Generator(device=pipe.device).manual_seed(seed)
    output = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        height=height,
        width=width,
        num_frames=num_frames,
        guidance_scale=guidance_scale,
        guidance_scale_2=guidance_scale_2,
        num_inference_steps=num_inference_steps,
        generator=generator,
        latents=noised_latents,
        callback_on_step_end=MultiPipelineCallbacks([attn_callback, blend_callback]),
        store=store,
        output_type="pil",
        start_noise_at=noise_step,
        refine_steps=refine_steps,
        refiner=refiner,
    ).frames[0]

    unregister_attention_hook(pipe, attn_processors)
    return output


def refine_video(
    pipe,
    prompt,
    negative_prompt,
    frames,
    height,
    width,
    num_frames,
    num_inference_steps,
    guidance_scale,
    guidance_scale_2,
    seed,
    noise_step,
):
    """Refine stage 1 output at higher resolution via SDEdit."""
    frames = [f.resize((width, height)) for f in frames]
    latents = encode_video_to_latents(pipe, video_to_tensor(frames))

    generator = torch.Generator(device=pipe.device).manual_seed(seed)
    noise = randn_tensor(latents.shape, generator, device=pipe.device, dtype=latents.dtype)
    pipe.scheduler.set_timesteps(num_inference_steps)
    t = torch.as_tensor(pipe.scheduler.timesteps[noise_step], device=pipe.device, dtype=torch.long).view(1)
    noised = pipe.scheduler.add_noise(latents, noise, t.long())

    generator = torch.Generator(device=pipe.device).manual_seed(seed)
    return pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        height=height,
        width=width,
        num_frames=num_frames,
        guidance_scale=guidance_scale,
        guidance_scale_2=guidance_scale_2,
        num_inference_steps=num_inference_steps,
        generator=generator,
        latents=noised,
        output_type="pil",
        start_noise_at=noise_step,
    ).frames[0]

## Configuration

In [ ]:
# Provide your own source video and edit mask
source_video_path = "path/to/source_video.mp4"
mask_path = "path/to/edit_mask.mp4"
output_path = os.path.join(OUTPUTS_PATH, "editing")

prompt = "A cheetah is running across the grass with a stick in its mouth."
negative_prompt = "色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，最差质量，低质量，JPEG压缩残留，丑陋的，残缺的，多余的手指，画得不好的手部，画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，静止不动的画面，杂乱的背景，三条腿，背景人很多，倒着走"
phrase_indices = [[1]]

num_frames = 121
fps = 24

stage1_height = 320
stage1_width = 480
stage1_steps = 25
stage1_guidance = 7.0
stage1_guidance_2 = 4.0
noise_step = 4
blend_start = 0
blend_stop = 15
layers_to_hook = list(range(0, 10))
num_refinements = 5
lr_start = 0.01
lr_end = 0.01
refine_steps = list(range(stage1_steps - noise_step))

stage2_height = 480
stage2_width = 720
stage2_steps = 20
stage2_noise_step = 7
stage2_guidance = 7.0
stage2_guidance_2 = 4.0

Path(output_path).mkdir(parents=True, exist_ok=True)

guiding_phrases, refiner = setup_layout_guidance(
    bbox_paths=[mask_path],
    phrase_indices=phrase_indices,
    prompt=prompt,
    tokenizer=pipe.tokenizer,
    height=stage1_height,
    width=stage1_width,
    num_frames=num_frames,
    num_refinements=num_refinements,
    lr_start=lr_start,
    lr_end=lr_end,
    vae_scale_factor_temporal=pipe.vae_scale_factor_temporal,
    vae_scale_factor_spatial=pipe.vae_scale_factor_spatial,
)

## Stage 1: Edit Source Video

In [ ]:
seed = 73

stage1_output = edit_video(
    pipe=pipe,
    prompt=prompt,
    negative_prompt=negative_prompt,
    source_video_path=source_video_path,
    mask_video_path=mask_path,
    height=stage1_height,
    width=stage1_width,
    num_frames=num_frames,
    num_inference_steps=stage1_steps,
    guidance_scale=stage1_guidance,
    guidance_scale_2=stage1_guidance_2,
    seed=seed,
    noise_step=noise_step,
    guiding_phrases=guiding_phrases,
    refiner=refiner,
    output_path=output_path,
    blend_start=blend_start,
    blend_stop=blend_stop,
    layers_to_hook=layers_to_hook,
    refine_steps=refine_steps,
)

stage1_path = Path(output_path) / "base.mp4"
export_to_video(stage1_output, str(stage1_path), fps=fps)
logger.success(f"Stage 1 saved: {stage1_path}")

## Stage 2: Refinement

In [ ]:
stage2_output = refine_video(
    pipe=pipe,
    prompt=prompt,
    negative_prompt=negative_prompt,
    frames=stage1_output,
    height=stage2_height,
    width=stage2_width,
    num_frames=num_frames,
    num_inference_steps=stage2_steps,
    guidance_scale=stage2_guidance,
    guidance_scale_2=stage2_guidance_2,
    seed=seed,
    noise_step=stage2_noise_step,
)

stage2_path = Path(output_path) / "final.mp4"
export_to_video(stage2_output, str(stage2_path), fps=fps)
logger.success(f"Stage 2 saved: {stage2_path}")